# FlashMTP Training Commands

`DFLASH_DISTILL_STAGE=none` 表示关闭 DFlash 蒸馏，只训练原始 FlashMTP CE loss。启用蒸馏时需要设置 `DFLASH_TEACHER_PATH`，并选择 `stage1` 或 `stage2`。


## DFlash 蒸馏参数说明

| 参数 | 含义 | 建议 |
|---|---|---|
| `DFLASH_TEACHER_PATH` | DFlash teacher checkpoint 路径；启用蒸馏时必须设置。 | 指向训练好的 DFlash teacher |
| `DFLASH_DISTILL_STAGE` | 蒸馏模式：`none` 关闭；`stage1` 只做 KL；`stage2` 做 CE + DFlash KL。 | v1.3 推荐 `stage2` |
| `DFLASH_DISTILL_WEIGHT` | DFlash KL loss 权重，即 $\\lambda_{kl}$。 | `1.0` 起步，stage2 可试 `0.3`/`1.0` |
| `DFLASH_DISTILL_TEMPERATURE` | KL 蒸馏温度，温度越高 teacher 分布越平滑。 | `2.0` |
| `DFLASH_DISTILL_TOP_K` | KL 只在 teacher top-k + true label 候选集上算，节省显存/算力。 | `128` |
| `DFLASH_CE_POS_MODE` | CE 位置策略：`all` 全部有效 slot；`prefix` 使用 teacher 连续正确前缀并在 student 追上后多监督一个错位。 | 推荐 `all`，配合 `DFLASH_CE_WRONG_WEIGHT` |
| `DFLASH_ALIGN_MODE` | 对齐模式：`final` 只对齐最终 logits；`final+mid` 额外对齐中间层 hidden。 | v1.3 推荐 `final+mid` |
| `DFLASH_MID_ALIGN` | 中间层对齐策略：`half` 只对齐中间一层；`full` 对齐除最后层外的所有层。 | 先用 `half` |
| `DFLASH_MID_WEIGHT` | 中间层 norm-hidden MSE 权重，即 $\\lambda_{mid}$；`0.0` 表示关闭。 | `1.0` 起步 |
| `DFLASH_CE_WEIGHT` | milestone 后 CE 最终上升到的目标权重，即 $\\lambda_{ce}$。 | `1.0` |
| `DFLASH_CE_MIN_SCALE` | CE 初始/最小保留比例；milestone 前 CE 权重为 `DFLASH_CE_WEIGHT * DFLASH_CE_MIN_SCALE`。 | `0.1`/`0.2` |
| `DFLASH_MILESTONE_EPOCH` | `final+mid` 的切换 epoch；之前 CE 使用 min scale，之后 KL/mid 余弦下降、CE 余弦上升。 | 例如总 6 epoch 设 `2`，总 12 epoch 设 `4` |
| `DFLASH_DISTILL_MIN_SCALE` | KL/mid 余弦衰减的最小保留比例；`0.2` 表示最终保留 20% 的蒸馏权重。 | `0.1`/`0.2` |
| `DFLASH_STAGE1_LOSS_DECAY_GAMMA` | stage1 使用的位置衰减 gamma。 | 默认 `14` |
| `DFLASH_STAGE2_LOSS_DECAY_GAMMA` | stage2 使用的位置衰减 gamma。 | 默认 `7` |

v1.3 的关键规则：KL 和 mid hidden 只蒸馏 DFlash top1 等于 true label 的 slot；CE 仍建议覆盖全部有效 slot；mid hidden 先 normalize 再做 MSE；milestone 后使用 cosine schedule 平滑从蒸馏切到 CE。`final` 与 `final+mid` 都使用该调度；区别是 `final` 没有 mid loss。


In [ ]:
%%bash
# Baseline: no DFlash distillation, CE only
cd /data/wanghanzhen/Projects/MTP/NIPS26/FlashMTP_v1.3
DFLASH_DISTILL_STAGE=none \
bash scripts/run_training_flashmtp.sh --dt qz


In [ ]:
# Stage 1: pure KL imitation of DFlash; DFlash correct/wrong slots all participate
cd /data/wanghanzhen/Projects/MTP/NIPS26/FlashMTP_v1.3
LOCAL_POSITION=true NUM_EPOCHS=12 \
DFLASH_TEACHER_PATH=$WHZ_DIR/models/z-lab/Qwen3-8B-DFlash-b16 \
DFLASH_DISTILL_STAGE=stage1 \
DFLASH_DISTILL_WEIGHT=1.0 \
DFLASH_DISTILL_TEMPERATURE=2.0 \
DFLASH_DISTILL_TOP_K=128 \
bash scripts/run_training_flashmtp.sh --dt h100


#### true-label CE + DFlash KL only on slots where DFlash top1 == true label


In [ ]:
cd /inspire/hdd/project/inference-chip/xujiaming-253308120313/whz/FlashMTP_v1.3
mkdir -p whz_mtp_logs
LOCAL_POSITION=true NUM_EPOCHS=8 BLOCK_SIZE=16 NUM_MIDDLE_LAYERS_N=16 \
DATA_NUM_SAMPLES=900000 MAX_LENGTH=4096 \
DFLASH_TEACHER_PATH=/inspire/hdd/project/inference-chip/xujiaming-253308120313/whz/FlashMTP/cache/models/dflash_mix_sample_14_think_off_qwen3_8b_maxlen20480_nnodes4/epoch_3_step_160000 \
TRAIN_DATA_PATH=/inspire/hdd/project/inference-chip/xujiaming-253308120313/whz/FlashMTP/cache/data/regen_data/mix_codealpaca_20k_nemotron_800k_orcamath_80k/merged_900k.jsonl \
DFLASH_ALIGN_MODE=final DFLASH_MILESTONE_EPOCH=2 \
LOSS_DECAY_GAMMA=7 DFLASH_DISTILL_DECAY_GAMMA=14 \
DFLASH_CE_POS_MODE=all DFLASH_CE_WEIGHT=1.0 \
DFLASH_DISTILL_WEIGHT=1.0 DFLASH_DISTILL_MIN_SCALE=0.2 \
DFLASH_DISTILL_TEMPERATURE=1.0 DFLASH_DISTILL_TOP_K=64 \
bash scripts/run_training_flashmtp.sh --dt qz \
   > "whz_mtp_logs/train_flashmtp_dist_$(date +%Y%m%d_%H%M%S).log" 2>&1


In [ ]:
cd /inspire/hdd/project/inference-chip/xujiaming-253308120313/whz/FlashMTP_v1.3
LOCAL_POSITION=true NUM_EPOCHS=12 BLOCK_SIZE=16 NUM_MIDDLE_LAYERS_N=16 \
DATA_NUM_SAMPLES=40000 MAX_LENGTH=4096 DFLASH_DISTILL_DECAY_GAMMA=0 \
DFLASH_TEACHER_PATH=/inspire/hdd/project/inference-chip/xujiaming-253308120313/whz/FlashMTP/cache/models/dflash_mix_sample_14_think_off_qwen3_8b_maxlen20480_nnodes4/epoch_3_step_160000 \
DFLASH_ALIGN_MODE=final DFLASH_MILESTONE_EPOCH=4 \
DFLASH_CE_POS_MODE=all \
DFLASH_DISTILL_STAGE=stage2 \
DFLASH_DISTILL_WEIGHT=1.0 DFLASH_CE_WEIGHT=0.8 DFLASH_DISTILL_MIN_SCALE=0.2 \
DFLASH_DISTILL_TEMPERATURE=2.0 \
DFLASH_DISTILL_TOP_K=128 \
bash scripts/run_training_flashmtp.sh --dt qz \
   > "whz_mtp_logs/train_dflash_origin_qz_dist_$(date +%Y%m%d_%H%M%S).log" 2>&1

#### v1.3 final+mid: DFlash KL + teacher-correct norm-hidden mid loss, then cosine switch to CE

推荐从这个配置开始跑：`stage2 + final+mid + half`。CE gate 保持 `all`，因为 KL/mid 已经只吸收 teacher 正确位置，CE 用全部有效 slot 纠偏。


In [ ]:
cd /data/wanghanzhen/Projects/MTP/NIPS26/FlashMTP_v1.3
# DFLASH_MID_WEIGHT needs scaling

LOCAL_POSITION=true NUM_EPOCHS=8 BLOCK_SIZE=16 NUM_MIDDLE_LAYERS_N=16 \
DATA_NUM_SAMPLES=40000 MAX_LENGTH=4096 NUM_ANCHORS=512 \
DFLASH_TEACHER_PATH=/inspire/hdd/project/inference-chip/xujiaming-253308120313/whz/FlashMTP/cache/models/dflash_mix_sample_14_think_off_qwen3_8b_maxlen20480_nnodes4/epoch_5_step_290000 \
DFLASH_MILESTONE_EPOCH=2 DFLASH_CE_WRONG_WEIGHT=0.2 \
LOSS_DECAY_GAMMA=7 DFLASH_DISTILL_DECAY_GAMMA=14 \
DFLASH_DISTILL_WEIGHT=1.0 DFLASH_DISTILL_MIN_SCALE=0.2 \
DFLASH_MID_WEIGHT=1000 DFLASH_ALIGN_MODE=final+mid DFLASH_MID_ALIGN=half \
DFLASH_CE_POS_MODE=all DFLASH_CE_WEIGHT=1.0 \
DFLASH_DISTILL_TEMPERATURE=1.0 DFLASH_DISTILL_TOP_K=64 \
bash scripts/run_training_flashmtp.sh --dt qz



#### v1.3 KL only，CE hearts

In [ ]:
CUDA_VISIBLE_DEVICES=2,3,4,5,6,7 \
NPROC_PER_NODE=6 \
LOCAL_POSITION=true NUM_EPOCHS=6 BLOCK_SIZE=16 NUM_MIDDLE_LAYERS_N=16 \
DATA_NUM_SAMPLES=40000 MAX_LENGTH=4096 NUM_ANCHORS=512 \
DFLASH_TEACHER_PATH=/data/wanghanzhen/Projects/MTP/NIPS26/FlashMTP_v1.3/cache/models/dflash_teacher \
DFLASH_MILESTONE_EPOCH=6 DFLASH_DISTILL_DECAY_GAMMA=14 \
DFLASH_DISTILL_WEIGHT=1.0 DFLASH_DISTILL_MIN_SCALE=0.2 \
DFLASH_ALIGN_MODE=final \
DFLASH_DISTILL_TEMPERATURE=1.0 DFLASH_DISTILL_TOP_K=64 \
DFLASH_CE_WEIGHT=0 \
bash scripts/run_training_flashmtp.sh --dt h100